In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
dataset_path = '/content/drive/My Drive/PlantVillageDataset/PlantVillage'

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, Activation


In [4]:
import os
import shutil
import random


dataset_path = '/content/drive/MyDrive/PlantVillageDataset/PlantVillage'

test_path = '/content/drive/MyDrive/PlantVillageDataset/PlantVillage_test'


if not os.path.exists(test_path):
    os.makedirs(test_path)

test_split = 0.20

for class_name in os.listdir(dataset_path):
    class_folder = os.path.join(dataset_path, class_name)
    if not os.path.isdir(class_folder):
        continue

    test_class_folder = os.path.join(test_path, class_name)
    if not os.path.exists(test_class_folder):
        os.makedirs(test_class_folder)

    images = os.listdir(class_folder)
    random.shuffle(images)

    num_test = int(len(images) * test_split)
    test_images = images[:num_test]

    for img in test_images:
        src = os.path.join(class_folder, img)
        dst = os.path.join(test_class_folder, img)
        shutil.move(src, dst)

print("Dataset successfully split into training and test set.")


Dataset successfully split into training and test set.


In [5]:
train_data_generator = ImageDataGenerator(
    rescale = 1/.255,
    rotation_range = 20,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.1,
    zoom_range = 0.1,
    horizontal_flip = True,
    fill_mode = 'nearest',
    validation_split = 0.2
)

train_generator = train_data_generator.flow_from_directory(
    dataset_path,
    target_size = (227,227),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'training'
)

val_generator = train_data_generator.flow_from_directory(
    dataset_path,
    target_size = (227,227),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'validation'
)

Found 11240 images belonging to 15 classes.
Found 2804 images belonging to 15 classes.


In [15]:
#AlexNet architecture defined

model = Sequential()

#1st convolution layer
model.add(Conv2D(96,kernel_size=(11,11),strides=(4,4),input_shape=(227,227,3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#2nd convolution layer
model.add(Conv2D(256,kernel_size=(5,5),padding='same'))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#3rd convolution layer
model.add(Conv2D(384,kernel_size=(3,3),padding='same'))
model.add(Activation('relu'))

#4th convolution layer
model.add(Conv2D(384,kernel_size=(3,3),padding='same'))
model.add(Activation('relu'))

#5th convolution layer
model.add(Conv2D(256,kernel_size=(3,3),padding='same'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#Flatten layer
model.add(Flatten())

#Dense layer 1
model.add(Dense(4096))
model.add(Activation('relu'))
model.add(Dropout(0.5))

#dense layer 2
model.add(Dense(4096))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# output layer

model.add(Dense(15))
model.add(Activation('softmax'))





/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
#compile the model
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [17]:
# save the model weights after each epoch
checkpoint_path = '/content/drive/My Drive/PlantVillageDataset/epoch{epoch:02d}.weights.h5'

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath = checkpoint_path,
    save_weights_only=True,
    save_best_only=False,
    verbose = 1
)


In [18]:
history = model.fit(train_generator,
                    validation_data=val_generator,
                    epochs = 3,
                    callbacks=[checkpoint_cb])

Epoch 1/3
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.1321 - loss: 71.2850 
Epoch 1: saving model to /content/drive/My Drive/PlantVillageDataset/epoch01.weights.h5
352/352 ━━━━━━━━━━━━━━━━━━━━ 6313s 18s/step - accuracy: 0.1321 - loss: 71.1286 - val_accuracy: 0.1633 - val_loss: 2.5525
Epoch 2/3
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - accuracy: 0.1654 - loss: 2.5541
Epoch 2: saving model to /content/drive/My Drive/PlantVillageDataset/epoch02.weights.h5
352/352 ━━━━━━━━━━━━━━━━━━━━ 267s 756ms/step - accuracy: 0.1654 - loss: 2.5541 - val_accuracy: 0.1854 - val_loss: 2.5059
Epoch 3/3
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 518ms/step - accuracy: 0.1868 - loss: 2.4895
Epoch 3: saving model to /content/drive/My Drive/PlantVillageDataset/epoch03.weights.h5
352/352 ━━━━━━━━━━━━━━━━━━━━ 230s 654ms/step - accuracy: 0.1868 - loss: 2.4894 - val_accuracy: 0.2086 - val_loss: 2.3313
